# SPARK Academy 2026 | Introduction to Machine Learning
## Notebook 1: ML Concepts & Supervised Learning

**Instructor:** Aondona Moses
**Dataset:** Pima Indians Diabetes Dataset

---

### What you will learn:
1. What is Machine Learning? Types and terminology
2. The ML workflow: data → train → evaluate → predict
3. **Linear Regression** — predicting continuous values
4. **Logistic Regression** — classifying patients (diabetic or not)
5. Model evaluation: accuracy, precision, recall, confusion matrix

### Dataset Context
This dataset is from the **National Institute of Diabetes and Digestive and Kidney Diseases (NIDDK)**. All patients are **Pima Indian females aged 21+**. The goal: predict whether a patient has diabetes based on clinical measurements.

| Feature | Description |
|---------|------------|
| Pregnancies | Number of pregnancies |
| Glucose | Plasma glucose concentration (oral glucose tolerance test) |
| BloodPressure | Diastolic blood pressure (mmHg) |
| SkinThickness | Triceps skin fold thickness (mm) |
| Insulin | 2-hour serum insulin (µU/ml) |
| BMI | Body mass index (kg/m²) |
| DiabetesPedigreeFunction | Genetic risk score |
| Age | Age in years |
| **Outcome** | **0 = No diabetes, 1 = Diabetes** |

---

## 1. Setup & Load Data

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report,
                             mean_squared_error, r2_score, roc_curve, auc)
import warnings
warnings.filterwarnings('ignore')

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

print("Libraries loaded!")

In [6]:
# Load the dataset
# Option 1: From Kaggle (if running on Kaggle)
# df = pd.read_csv("/kaggle/input/diabetes-data-set/diabetes.csv")

# Option 2: Direct URL
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
           "Insulin", "BMI", "DiabetesPedigreeFunction", "Age", "Outcome"]
df = pd.read_csv(url, names=columns)

print(f"Dataset shape: {df.shape}")
print(f"Patients: {df.shape[0]}, Features: {df.shape[1] - 1}")
df.head(10)

## 2. What is Machine Learning?

**Machine Learning** is teaching a computer to learn patterns from data, instead of explicitly programming rules.

### Three Types of ML:

| Type | What it does | Example |
|------|-------------|---------|
| **Supervised Learning** | Learn from labeled data (input → output) | Predict diabetes from clinical features |
| **Unsupervised Learning** | Find patterns in unlabeled data | Group patients by similarity |
| **Reinforcement Learning** | Learn through trial and error | Robot surgery planning |

### Key Terminology:

| Term | Meaning |
|------|---------|
| **Features (X)** | Input variables (Glucose, BMI, Age, etc.) |
| **Target (y)** | What we want to predict (Outcome: 0 or 1) |
| **Training set** | Data used to teach the model |
| **Test set** | Data used to evaluate (never seen during training) |
| **Overfitting** | Model memorizes training data, fails on new data |
| **Underfitting** | Model is too simple, misses patterns |

## 3. Explore the Data

Before building any model, understand your data. This is the **most important step**.

In [7]:
# Basic stats
print("=== Dataset Info ===")
print(f"Shape: {df.shape}")
print(f"\nTarget distribution:")
print(df["Outcome"].value_counts())
print(f"\nDiabetes rate: {df['Outcome'].mean():.1%}")

In [ ]:
# Summary statistics
df.describe().round(2)

In [ ]:
# Check for zeros in columns where 0 is impossible
# (You can't have 0 Glucose, 0 BloodPressure, etc.)
zero_cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
print("Zeros in columns (likely missing values):")
for col in zero_cols:
    n_zeros = (df[col] == 0).sum()
    pct = n_zeros / len(df) * 100
    print(f"  {col}: {n_zeros} ({pct:.1f}%)")

In [ ]:
# Replace zeros with NaN, then fill with median
df_clean = df.copy()
for col in zero_cols:
    df_clean[col] = df_clean[col].replace(0, np.nan)
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

print("After cleaning:")
print(df_clean.describe().round(2))

In [ ]:
# Visualize distributions
fig, axes = plt.subplots(3, 3, figsize=(14, 10))
fig.suptitle("Feature Distributions by Diabetes Outcome", fontsize=14, fontweight="bold")

for i, col in enumerate(df_clean.columns[:-1]):
    ax = axes[i // 3][i % 3]
    df_clean[df_clean["Outcome"] == 0][col].hist(ax=ax, alpha=0.6, label="No Diabetes", color="#3498DB", bins=20)
    df_clean[df_clean["Outcome"] == 1][col].hist(ax=ax, alpha=0.6, label="Diabetes", color="#E74C3C", bins=20)
    ax.set_title(col, fontweight="bold")
    ax.legend(fontsize=8)

axes[2][2].axis("off")
plt.tight_layout()
plt.show()

print("Key observation: Glucose, BMI, and Age show clear separation between groups")

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10, 8))
corr = df_clean.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdYlBu_r",
            center=0, square=True, linewidths=0.5)
plt.title("Feature Correlations", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("Strongest correlations with Outcome:")
print(corr["Outcome"].drop("Outcome").sort_values(ascending=False).to_string())

## 4. The ML Workflow

Every ML project follows this pipeline:

```
1. Prepare Data → 2. Split Train/Test → 3. Train Model → 4. Evaluate → 5. Predict
```

In [ ]:
# Step 1: Separate features and target
X = df_clean.drop("Outcome", axis=1)
y = df_clean["Outcome"]

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Feature names: {X.columns.tolist()}")

In [ ]:
# Step 2: Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} patients ({X_train.shape[0]/len(X):.0%})")
print(f"Test set: {X_test.shape[0]} patients ({X_test.shape[0]/len(X):.0%})")
print(f"\nTraining diabetes rate: {y_train.mean():.1%}")
print(f"Test diabetes rate: {y_test.mean():.1%}")
print("\nstratify=y ensures both sets have the same diabetes ratio!")

In [ ]:
# Step 3: Scale features (important for many ML algorithms)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit on train, transform train
X_test_scaled = scaler.transform(X_test)          # only transform test (no fit!)

print("Before scaling:")
print(f"  Glucose mean: {X_train['Glucose'].mean():.1f}, std: {X_train['Glucose'].std():.1f}")
print(f"  Age mean: {X_train['Age'].mean():.1f}, std: {X_train['Age'].std():.1f}")
print("\nAfter scaling:")
print(f"  Glucose mean: {X_train_scaled[:, 1].mean():.4f}, std: {X_train_scaled[:, 1].std():.4f}")
print(f"  Age mean: {X_train_scaled[:, 7].mean():.4f}, std: {X_train_scaled[:, 7].std():.4f}")
print("\nAll features now have mean ~0 and std ~1")

## 5. Linear Regression

**Linear Regression** predicts a continuous value by fitting a line through the data.

**Formula:** y = w₁x₁ + w₂x₂ + ... + w₈x₈ + b

In our context: we predict a continuous risk score (not ideal for classification, but important to understand).

### When to use:
- Predicting a **number** (tumor size, blood pressure, survival time)
- Understanding **which features matter most** (via coefficients)

In [ ]:
# Linear Regression: predicting Glucose from other features
# (Demonstrating regression with a continuous target)
X_reg = df_clean.drop(["Glucose", "Outcome"], axis=1)
y_reg = df_clean["Glucose"]

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

# Train
lin_reg = LinearRegression()
lin_reg.fit(X_train_reg, y_train_reg)

# Predict
y_pred_reg = lin_reg.predict(X_test_reg)

# Evaluate
mse = mean_squared_error(y_test_reg, y_pred_reg)
r2 = r2_score(y_test_reg, y_pred_reg)

print("=== Linear Regression: Predicting Glucose Level ===")
print(f"Mean Squared Error: {mse:.2f}")
print(f"R² Score: {r2:.4f}")
print(f"\nFeature importance (coefficients):")
for name, coef in sorted(zip(X_reg.columns, lin_reg.coef_), key=lambda x: abs(x[1]), reverse=True):
    print(f"  {name:30s} {coef:+.4f}")

In [ ]:
# Visualize predictions vs actual
plt.figure(figsize=(8, 6))
plt.scatter(y_test_reg, y_pred_reg, alpha=0.5, color="#3498DB", edgecolors="white", s=60)
plt.plot([y_test_reg.min(), y_test_reg.max()], [y_test_reg.min(), y_test_reg.max()],
         'r--', linewidth=2, label="Perfect prediction")
plt.xlabel("Actual Glucose", fontsize=12)
plt.ylabel("Predicted Glucose", fontsize=12)
plt.title("Linear Regression: Actual vs Predicted Glucose", fontsize=14, fontweight="bold")
plt.legend()
plt.tight_layout()
plt.show()

## 6. Logistic Regression

**Logistic Regression** is for **classification** — predicting a category (diabetic or not).

Despite the name, it is a **classification** algorithm, not regression!

**How it works:** Instead of fitting a line, it fits an S-shaped curve (sigmoid) that outputs a probability between 0 and 1.

- If probability > 0.5 → predict class 1 (diabetic)
- If probability ≤ 0.5 → predict class 0 (not diabetic)

### When to use:
- **Binary classification** (yes/no, positive/negative, diabetic/healthy)
- When you need **probability** of belonging to a class
- When you want an **interpretable** model

In [ ]:
# Logistic Regression: predicting diabetes
log_reg = LogisticRegression(random_state=42, max_iter=1000)
log_reg.fit(X_train_scaled, y_train)

# Predict
y_pred = log_reg.predict(X_test_scaled)
y_prob = log_reg.predict_proba(X_test_scaled)[:, 1]  # probability of diabetes

# Evaluate
print("=== Logistic Regression: Diabetes Prediction ===")
print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall: {recall_score(y_test, y_pred):.4f}")
print(f"F1 Score: {f1_score(y_test, y_pred):.4f}")

In [ ]:
# Classification report
print("=== Detailed Classification Report ===")
print(classification_report(y_test, y_pred, target_names=["No Diabetes", "Diabetes"]))

### Understanding Evaluation Metrics

| Metric | What it measures | Medical meaning |
|--------|-----------------|----------------|
| **Accuracy** | % of all predictions that are correct | Overall correctness |
| **Precision** | Of those predicted diabetic, how many actually are? | Avoid false alarms |
| **Recall** | Of actual diabetic patients, how many did we catch? | Don’t miss sick patients! |
| **F1 Score** | Balance between precision and recall | Overall performance |

In **medical AI**, recall is often more important than precision — missing a diabetic patient is worse than a false alarm.

In [ ]:
# Confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=["No Diabetes", "Diabetes"],
            yticklabels=["No Diabetes", "Diabetes"])
axes[0].set_ylabel("Actual", fontsize=12)
axes[0].set_xlabel("Predicted", fontsize=12)
axes[0].set_title("Confusion Matrix", fontsize=14, fontweight="bold")

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color="#E74C3C", linewidth=2, label=f"ROC Curve (AUC = {roc_auc:.3f})")
axes[1].plot([0, 1], [0, 1], "k--", alpha=0.5, label="Random (AUC = 0.5)")
axes[1].set_xlabel("False Positive Rate", fontsize=12)
axes[1].set_ylabel("True Positive Rate (Recall)", fontsize=12)
axes[1].set_title("ROC Curve", fontsize=14, fontweight="bold")
axes[1].legend(fontsize=11)
axes[1].set_xlim([0, 1])
axes[1].set_ylim([0, 1.05])

plt.tight_layout()
plt.show()

print(f"AUC = {roc_auc:.3f}")
print("AUC > 0.7 = acceptable, > 0.8 = good, > 0.9 = excellent")

In [ ]:
# Feature importance for Logistic Regression
importance = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": log_reg.coef_[0]
}).sort_values("Coefficient", key=abs, ascending=True)

plt.figure(figsize=(8, 5))
colors = ["#E74C3C" if c > 0 else "#3498DB" for c in importance["Coefficient"]]
plt.barh(importance["Feature"], importance["Coefficient"], color=colors)
plt.xlabel("Coefficient (positive = increases diabetes risk)", fontsize=11)
plt.title("Logistic Regression: Feature Importance", fontsize=14, fontweight="bold")
plt.axvline(x=0, color="black", linewidth=0.8)
plt.tight_layout()
plt.show()

print("Red = increases diabetes risk, Blue = decreases risk")
print("Glucose has the strongest influence on diabetes prediction")

## 7. Key Takeaways

1. **Linear Regression** predicts continuous values (numbers). Use for predicting measurements.
2. **Logistic Regression** classifies into categories. Use for yes/no, positive/negative decisions.
3. Always **split your data** into train and test sets before training.
4. Always **scale your features** when using algorithms sensitive to magnitude.
5. In medical AI, **recall** (catching all sick patients) is often more important than precision.
6. The **confusion matrix** and **ROC curve** give you a complete picture of model performance.

### What’s Next:
- **Notebook 2:** Unsupervised Learning (K-means, Hierarchical Clustering)
- **Notebook 3:** Full Implementation Pipeline with Multiple Models

---
*SPARK Academy 2026 — Train for Change, From Science to Practice* 🚀